# 04 · Fine-tune the GPT-2 doctor

**SimClinic, CS425 Group 7 (doctor model).** This notebook fine-tunes GPT-2 on the doctor training strings from notebook 03, over the grid in the proposal:
- **model size:** GPT-2 small (124M) and medium (355M);
- **learning rate:** 5e-5 and 1e-4;
- up to 5 epochs with early stopping on dev loss, an effective batch of 16 through gradient accumulation, and the full 1,024-token context.

**Selection.** Each run is scored in auto mode on a 50-case dev set: the doctor interviews the rule-based card patient, and the run with the best dev top-3 accuracy is kept (ties broken by top-1, then dev loss). The proposal's selection rule uses validation loss plus a dev-set score, and for the doctor that score is diagnostic accuracy.

Outputs:
- `checkpoints/doctor/<model>-lr<lr>/best/`: one checkpoint per run;
- `checkpoints/doctor/best/`: the selected doctor, with `doctor_config.json` (candidate diseases, run settings, scores);
- `checkpoints/doctor/grid_results.csv`.

## 0. Config

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

MODELS = ["gpt2", "gpt2-medium"]
LEARNING_RATES = [5e-5, 1e-4]
MAX_EPOCHS = 5
PATIENCE = 1                       # stop when dev loss does not improve for this many epochs
EFFECTIVE_BATCH = 16
PER_DEVICE_BATCH = {"gpt2": 8, "gpt2-medium": 2}    # fits a 12 GB GPU at the longest dialogue (458 tokens);
                                                   # lower it on smaller cards (gradient accumulation keeps the batch at 16)
WARMUP_SHARE = 0.05
DEV_CASES = 50                     # auto-mode dev set for model selection
MAX_QUESTIONS = 10

DATA_DIR = ROOT / "data" / "processed" / "doctor"
TOK_DIR = DATA_DIR / "tokenized" / "gpt2"
CKPT_DIR = ROOT / "checkpoints" / "doctor"
SEED = 42

## 1. Load data

In [2]:
import gc
import json
import math
import shutil
import pandas as pd
import torch
from datasets import load_from_disk
from transformers import (AutoModelForCausalLM, AutoTokenizer, DataCollatorForSeq2Seq, EarlyStoppingCallback,
                          Trainer, TrainingArguments, set_seed)
from simclinic.lexicon import Lexicon
from simclinic.naive_bayes import SymcatNaiveBayes

ds = load_from_disk(str(TOK_DIR / "dataset")).remove_columns("dialogue_id")
tokenizer = AutoTokenizer.from_pretrained(str(TOK_DIR / "tokenizer"))
prep = json.loads((TOK_DIR / "prep_config.json").read_text())
collator = DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100)

CONDITIONS = json.loads((DATA_DIR / "conditions.json").read_text())
symcat_table = json.loads((DATA_DIR / "symcat_conditions.json").read_text())
symptom_vocab = sorted({s for n in CONDITIONS for s in symcat_table[n]["symptoms"]})
lexicon = Lexicon.load(symptoms=symptom_vocab)
nb = SymcatNaiveBayes({n: symcat_table[n]["symptoms"] for n in CONDITIONS})

dev_cards = [json.loads(l) for l in (DATA_DIR / "cards" / "dev.jsonl").read_text().splitlines()]
# One card per disease first, so 50 cases cover 50 different diseases.
rank_in_disease, seen = [], {}
for c in dev_cards:
    seen[c["disease"]] = seen.get(c["disease"], -1) + 1
    rank_in_disease.append(seen[c["disease"]])
dev_cases = [c for _, c in sorted(zip(rank_in_disease, dev_cards), key=lambda x: x[0])][:DEV_CASES]

device = "cuda" if torch.cuda.is_available() else "cpu"
bf16 = device == "cuda" and torch.cuda.is_bf16_supported()
print({s: len(ds[s]) for s in ds}, "| device:", device, "| bf16:", bf16, "| dev cases:", len(dev_cases),
      f"covering {len({c['disease'] for c in dev_cases})} diseases")

{'train': 4827, 'dev': 201, 'test': 201} | device: cuda | bf16: True | dev cases: 50 covering 50 diseases


## 2. Auto-mode dev score

In [3]:
from simclinic.consult import run_consultation, score
from simclinic.doctors import GPT2Doctor
from simclinic.patients import CardPatient

def auto_mode(doctor, cases, seed=SEED):
    rows = []
    for i, card in enumerate(cases):
        set_seed(seed + i)
        rec = run_consultation(doctor, CardPatient(card, lexicon, seed=seed + i), max_questions=MAX_QUESTIONS)
        rows.append({"card_id": card["card_id"], "disease": card["disease"], **score(rec, card, nb, lexicon),
                     "record": rec})
    return pd.DataFrame(rows)

def dev_score(model):
    df = auto_mode(GPT2Doctor(model, tokenizer, CONDITIONS), dev_cases)
    return {"dev_top1": df["top1"].mean(), "dev_top3": df["top3"].mean(), "dev_questions": df["n_questions"].mean(),
            "dev_forced": df["forced"].mean()}

## 3. Train one configuration
New embedding rows for the special tokens are initialised to the mean of the existing ones (`resize_token_embeddings` does this by default). Mid-epoch checkpoints are not kept. After training, the best epoch (by dev loss) is reloaded, saved, and scored in auto mode. A finished run writes `run.json`, and re-running the notebook skips it, so an interrupted grid resumes.

In [4]:
def train_one(model_name, lr):
    run_dir = CKPT_DIR / f"{model_name}-lr{lr:g}"
    if (run_dir / "run.json").exists():
        return json.loads((run_dir / "run.json").read_text())
    set_seed(SEED)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    model.resize_token_embeddings(len(tokenizer))
    model.config.pad_token_id = tokenizer.pad_token_id
    bs = PER_DEVICE_BATCH[model_name]
    steps_per_epoch = math.ceil(len(ds["train"]) / EFFECTIVE_BATCH)
    args = TrainingArguments(
        output_dir=str(run_dir), learning_rate=lr, num_train_epochs=MAX_EPOCHS,
        per_device_train_batch_size=bs, per_device_eval_batch_size=bs,
        gradient_accumulation_steps=EFFECTIVE_BATCH // bs,
        warmup_steps=int(WARMUP_SHARE * steps_per_epoch * MAX_EPOCHS), weight_decay=0.01,
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1, save_only_model=True,
        load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
        logging_steps=50, report_to="none", seed=SEED, bf16=bf16, fp16=device == "cuda" and not bf16,
    )
    trainer = Trainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["dev"],
                      data_collator=collator, callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])
    result = trainer.train()
    eval_loss = trainer.evaluate()["eval_loss"]
    best = run_dir / "best"
    trainer.save_model(str(best))
    tokenizer.save_pretrained(str(best))
    for ckpt in run_dir.glob("checkpoint-*"):
        shutil.rmtree(ckpt)
    curve = [(round(h["epoch"]), h["eval_loss"]) for h in trainer.state.log_history if "eval_loss" in h]
    row = {"model": model_name, "lr": lr, "epochs_run": round(trainer.state.epoch, 2),
           "best_epoch": min(curve, key=lambda x: x[1])[0], "dev_loss": eval_loss,
           "train_minutes": result.metrics["train_runtime"] / 60, **dev_score(trainer.model),
           "path": str(best.relative_to(ROOT)), "curve": curve}
    (run_dir / "run.json").write_text(json.dumps(row, indent=1))
    del trainer, model
    gc.collect()                 # Trainer holds reference cycles; free GPU memory before the next run
    torch.cuda.empty_cache()
    return row

## 4. Grid search

In [5]:
rows = []
for model_name in MODELS:
    for lr in LEARNING_RATES:
        rows.append(train_one(model_name, lr))
        print({k: v for k, v in rows[-1].items() if k != "curve"})

grid = pd.DataFrame(rows)
grid.drop(columns="curve").to_csv(CKPT_DIR / "grid_results.csv", index=False)
grid.drop(columns=["curve", "path"]).round(4)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,0.903507,0.796605
2,0.753034,0.711299
3,0.698450,0.683986
4,0.666963,0.668482
5,0.655602,0.660914


[W1028 01:03:41.271890987 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 595591168 bytes (free: 299696128, total: 12533104640).


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (44 outputs left out)

[W1028 01:42:22.590493502 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 767557632 bytes (free: 308150272, total: 12533104640).


[W1028 01:42:52.681156752 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 792723456 bytes (free: 404684800, total: 12533104640).


{'model': 'gpt2-medium', 'lr': 0.0001, 'epochs_run': 5.0, 'best_epoch': 4, 'dev_loss': 0.6146042943000793, 'train_minutes': 13.079088333333335, 'dev_top1': np.float64(0.7), 'dev_top3': np.float64(0.88), 'dev_questions': np.float64(6.66), 'dev_forced': np.float64(0.0), 'path': 'checkpoints/doctor/gpt2-medium-lr0.0001/best'}


,model,lr,epochs_run,best_epoch,dev_loss,train_minutes,dev_top1,dev_top3,dev_questions,dev_forced
0,gpt2,0.0000,5.0,5,0.6609,5.0027,0.52,0.84,6.32,0.0
1,gpt2,0.0001,5.0,5,0.6306,4.9973,0.64,0.92,6.10,0.0
2,gpt2-medium,0.0000,5.0,5,0.6122,13.1213,0.60,0.88,6.44,0.0
3,gpt2-medium,0.0001,5.0,4,0.6146,13.0791,0.70,0.88,6.66,0.0


In [6]:
# Dev loss per epoch for each run (early stopping ends a run after PATIENCE epochs without improvement).
pd.DataFrame({f"{r['model']} lr={r['lr']:g}": dict(r["curve"]) for r in rows}).rename_axis("epoch").round(4)

,gpt2 lr=5e-05,gpt2 lr=0.0001,gpt2-medium lr=5e-05,gpt2-medium lr=0.0001
epoch,,,,
1,0.7966,0.7453,0.7023,0.6863
2,0.7113,0.6790,0.6436,0.6300
3,0.6840,0.6539,0.6255,0.6183
4,0.6685,0.6362,0.6136,0.6146
5,0.6609,0.6306,0.6122,0.6146


## 5. Select and save the best doctor

In [7]:
best = grid.sort_values(["dev_top3", "dev_top1", "dev_loss"], ascending=[False, False, True]).iloc[0]
best_dir = CKPT_DIR / "best"
if best_dir.exists():
    shutil.rmtree(best_dir)
shutil.copytree(ROOT / best["path"], best_dir)
doctor_config = {"base_model": best["model"], "learning_rate": float(best["lr"]), "candidates": CONDITIONS,
                 "max_questions": MAX_QUESTIONS, "data": prep["dialogues_source"],
                 "scores": {k: float(best[k]) for k in ["dev_loss", "dev_top1", "dev_top3", "dev_questions"]}}
(best_dir / "doctor_config.json").write_text(json.dumps(doctor_config, indent=2))
print(f"best: {best['model']} lr={best['lr']:g} -> {best_dir.relative_to(ROOT)}")
pd.Series(doctor_config["scores"]).round(3).to_frame("value")

best: gpt2 lr=0.0001 -> checkpoints/doctor/best


,value
dev_loss,0.631
dev_top1,0.640
dev_top3,0.920
dev_questions,6.100


## 6. Sample consultation with the selected doctor

In [8]:
doctor = GPT2Doctor.from_pretrained(str(best_dir), CONDITIONS)
card = dev_cases[3]
set_seed(SEED)
rec = run_consultation(doctor, CardPatient(card, lexicon, seed=SEED), max_questions=MAX_QUESTIONS)
print(f"hidden: {card['disease']} | card: {', '.join(card['symptoms'])}\n")
for t in rec["turns"]:
    print(f"{t['role']:>8}: {t['text']}")
print(f"\n  top-3: {'; '.join(rec['diagnosis'])}  ({rec['n_questions']} questions{', forced' if rec['forced'] else ''})")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

hidden: Acute sinusitis | card: Cough, Headache, Coryza, Ear pain

  doctor: What brings you in today?
 patient: Doctor, I'm quite worried. I've had a cough and a headache, and it started yesterday.
  doctor: Can you tell me more about the cough?
 patient: Yes, I've had a cough. Should I be worried?
  doctor: Do you have any pain in your ears or face?
 patient: Yes, I've had ear pain. Is it something serious?
  doctor: Do you have any ear pain in your head or neck?
 patient: Yes, I've had ear pain.
  doctor: Have you noticed any fever or a runny nose?
 patient: I've had a runny nose, but I haven't had a fever.
  doctor: Is there anything that makes the cough better or worse?
 patient: Yes, I've had a cough. Is it something serious?
  doctor: I understand you're worried. Let's keep an eye on your symptoms and see if they improve.

  top-3: Acute sinusitis; Acute otitis media; Common cold  (6 questions)


## 7. Using the doctor in the app
```python
from simclinic.doctors import GPT2Doctor
cfg = json.load(open("checkpoints/doctor/best/doctor_config.json"))
doctor = GPT2Doctor.from_pretrained("checkpoints/doctor/best", cfg["candidates"])   # CPU works too
turn = doctor.next_turn(history)          # history: [{"role": "doctor"|"patient", "text": ...}, ...]
if turn.diagnosis:                        # the doctor closed the consultation
    show(turn.text, turn.diagnosis)       # top-3 differential, best first
else:
    ask(turn.text)
# force a diagnosis (e.g. a "Diagnose now" button):  doctor.next_turn(history, force_diagnosis=True)
```
`05_doctor_eval.ipynb` compares this doctor with the baselines on the held-out test cards.